# COGS 188 - Project Proposal (Group Name: Rat-Brained)
## Prioritized Replay Meets Predictive Maps: A Hippocampus-Inspired Agent for Rapid Adaptation in Changing Environments

# Names

- Elizabeth Kao
- Skylar Belsher
- Siddhant Bhardwaj

### GitHub repo
- https://github.com/skyebel/COGS188_Hippocampus-Inspired-Agent-for-Rapid-Adaptation

# Abstract

Animals can adapt quickly when a reward moves or a path gets blocked, but many deep RL agents need many more trials. In this project we test if a brain-inspired agent can adapt faster. Our agent combines successor representations (SR), which work like a map of future states, with Dyna-style replay that picks which memories to replay using a "need x gain" rule. Rather than using a pre-existing dataset, we generate data by simulation in large multi-room mazes of around 25x25 to 30x30 cells, where the agent slips 10% of the time. After training, we change the reward, rewire the maze, or block/open a path. We compare Q-learning, Dyna-Q, a model-based planner, SR-TD, SR-Dyna, and SR with prioritized replay across 50+ random seeds. We evaluate success using two categories of metrics. The first category assesses behavior and computational efficiency by tracking the number of episodes needed to reach 90% of the way from random to optimal post-change performance, the area under the post-change learning curve, the online regret relative to the optimal policy, and the total number of replay updates. The second category involves neurobiological checks to confirm whether the model mimics biological brains, specifically by measuring the fraction of forward versus reverse replay events and determining if the learned successor representation features resemble hippocampal place fields.

# Background


Reinforcement learning methods usually fall into two groups. Model-free methods like Q-learning are cheap to run, but they adapt slowly when the environment changes. Model-based methods plan ahead using a model of the world, so they adapt right away, but planning takes a lot of computation<a name="sutton"></a>[<sup>[1]</sup>](#suttonnote). Animals seem to fall somewhere in between. The successor representation (SR) is one way to bridge the two. The SR stores how often the agent expects to visit each state in the future, so when a reward changes, the agent can update its values right away without relearning the layout of the maze<a name="dayan"></a>[<sup>[2]</sup>](#dayannote). Human behavior fits this idea. People adapt quickly when rewards change but have more trouble when the layout changes<a name="momennejad"></a>[<sup>[3]</sup>](#momennejadnote). Place cells in the hippocampus also behave a lot like SR features<a name="stachenfeld"></a>[<sup>[4]</sup>](#stachenfeldnote), and other work argues that the SR connects model-free and model-based learning along a scale rather than treating them as two separate systems<a name="russek"></a>[<sup>[5]</sup>](#russeknote).


The brain also keeps learning offline. When animals rest or pause, hippocampal place cells replay paths they have taken before, and right after getting a reward, they often replay the path in reverse<a name="foster"></a>[<sup>[6]</sup>](#fosternote). Complementary Learning Systems theory proposes that this replay helps train a slower learning system in the cortex<a name="cls"></a>[<sup>[7]</sup>](#clsnote). Mattar and Daw showed that an agent that replays the memories with the highest expected benefit, using a “need x gain” rule, reproduces many replay patterns seen in animals, including when replay runs forward and when it runs in reverse<a name="mattar"></a>[<sup>[8]</sup>](#mattarnote). In RL, Dyna is the standard way to use replay for planning<a name="dyna"></a>[<sup>[9]</sup>](#dynanote). All of our agents learn using temporal-difference (TD) learning, which resembles the way dopamine neurons signal reward prediction errors<a name="schultz"></a>[<sup>[10]</sup>](#schultznote). More recent work shows that dopamine neurons encode a full range of possible rewards, not just a single average<a name="dabney"></a>[<sup>[11]</sup>](#dabneynote).


Each of these ideas has been studied on its own, but in our literature review we did not find a study that compares them side by side with equal compute, using both engineering and neuroscience metrics. We want to find out which components help an agent adapt quickly, and whether brain-like replay patterns appear on their own from a simple utility rule. Answering these questions can help build agents that learn from less experience and can test theories of how the hippocampus works.

# Problem Statement

As mentioned prior in our proposal, animals adapt rapidly when a reward changes or a path alters, but standard RL agents do not. Model-free TD learners (like Q-learning) are cheap to run, but must re-experience a change to correct their stored values. Model-based planners adapt rapidly, but require heavy computation to recompute values from a learned model [<sup>[1]</sup>](#suttonnote), unlike biological brains. The successor representation (SR) acts as a compromise or intermediary synthesis between model-free TD learners and model-based planners. SR agents can handle a change in reward without relearning a maze, yet cannot handle a change in maze layout without further updating [<sup>[5]</sup>](#russeknote). It is not yet clear if prioritized offline replay can close the gap of layout modification when the amount of computation is fixed.

We plan to train various traditional and non-traditional agents in maze tasks until they reach a performance criterion, then either change the reward (reward revaluation) or change the maze layout (transition revaluation and detour tasks). Each agent is allowed $k$ replay backups per environment step. We ask whether an SR agent whose replay is ranked by need x gain [<sup>[8]</sup>](#mattarnote) needs fewer post-change episodes to recover than Q-learning, SR-TD, and uniform-replay agents using the same number of backups, and whether it gets close to a model-based planner in terms of rapid adaptation.
### Scientific Basis
- **Quantifiable:** The mazes are finite MDPs, so the optimal post-change policy and return can be computed exactly with value iteration. Recovery is the number of post-change episodes $\tau$ needed to reach a fixed fraction of that optimal return.
- **Measurable:** Every run logs post-change returns, recovery time, regret, the number of backups used, and each replay event.
- **Replicable:** The mazes are Gymnasium-compatible, every agent and task is run over 50+ random seeds, and all settings are stored in configuration files.

### Hypotheses
- **H1:** SR agents recover from reward revaluation faster than Q-learning, but show no advantage on transition revaluation unless replay is used to update the SR.
- **H2:** Adding need x gain replay to the SR agent closes the transition revaluation gap and approaches the model-based planner while fewer backups.
- **H3:** Replay will be mostly reverse right after a reward and mostly forward before a decision, as in Mattar and Daw [<sup>[8]</sup>](#mattarnote).
- **H4 (exploratory, if time permits):** Prioritization matters more than the raw amount of replay when the replay budget is small.

# Data


We are not using an existing dataset. All of our data will come from simulation in maze environments we build ourselves so that each agent interacts with them through the same interface. The code will be in our project repository. Every run uses a fixed random seed, which controls both the agent’s random choices and the environment’s random slips, so any result can be reproduced exactly.


### Environments


The mazes are multi-room layouts between 25x25 and 30x30 cells, roughly 600 to 900 open states after walls are removed, with doorways connecting the rooms. Each episode starts with the agent in a start cell and ends when it reaches a goal or hits the step limit. Rewards are only given at goal states. Movement in the maze is stochastic. When the agent chooses a direction, it moves that way 90% of the time. The other 10% of the time, it slips into a random neighboring cell instead. This means the agent cannot just memorize a fixed sequence of moves and has to learn how the maze actually behaves.


We use mazes because the tasks we are modeling usually run in mazes, which make our results easier to compare with the animal and human studies. We will build three tasks, each with a training phase and a change phase:


- Reward revaluation: the maze has two goals with different reward values. After training, we swap the values, so the agent’s preferred goal becomes the worse one. The layout does not change.
- Transition revaluation: after training, we change how part of the maze connects, for example by moving a doorway so a corridor leads to a different room. The rewards do not change.
- Tolman detour: after training, we block a doorway on the agent’s usual route, or open a new shortcut, and see whether the agent finds the new best path.


We will use one fixed layout per task so that differences between runs come from the agents and the random seed, not from the maze. If we have enough time, we will also test on a few extra layouts to check that our results are not specific to one maze, and add keys and doors to make the planning problem harder.


### Observations and size


Each observation is one step of an agent in an environment, recorded as (state, action, reward, next state, done) along with the agent, task, seed, phase (training or post-change), episode, and step number.


In each run, the agent trains until it reaches a performance criterion, up to a maximum of 500 episodes. We plan to define the criterion as reaching the goal in at least 90% of the last 20 episodes, though we may adjust this after testing. We then change the environment and run 200 more episodes. Each episode is capped at 1,000 steps, which is about 10 times the shortest path through the largest maze. This gives early agents room to explore without letting them wander forever. Once agents have learned the maze, most episodes take 100 to 150 steps, so a single run produces roughly 50,000 to 100,000 transitions. Across 50+ seeds, 6 agents, 3 tasks, and our ablations, this comes to tens of millions of transitions.


We will confirm these numbers with a short pilot run in week 2. The agent with need x gain replay is the main concern, since it has to evaluate many possible backups at every real step. If it happens to be too slow, we'll lower the number of post-change episodes or replay budget.


### Variables


The main variables are:


- State: the index of the agent’s current cell, one-hot encoded as input to the SR
- Action: one of four moves (N, S, E, W)
- Reward: the value received at goal states, which we change in the reward revaluation task
- Transition structure: which cells connect to which, which we change in the transition revaluation and detour tasks


We will also log what happens inside each agent. This includes the SR matrix M, the number of replay updates, and a record of each replay event, which notes the state and action that were replayed, the direction of the replay, and when it happened. This is important for neuroscience metrics, such as the fraction of forward vs. reverse replay and whether SR features look like place fields.


### Handling and storage


Since the data is simulated, there is nothing to clean, but we will put a system in place to keep it organized and standardized.


We will tune hyperparameters on about 10 seeds and report final results on a separate set of 50+ seeds, so our results are not inflated by tuning on the same runs we report.


Before running full experiments, we will test the environments and agents on small 5x5 mazes where we know the correct answers. The slip rate should come out close to 10%, the planner should find the optimal policy, and the learned SR should match the SR calculated directly from the transition probabilities.


Saving every step for every run would produce a very large dataset. Instead, each run will get a table with one row per episode, recording steps taken, total reward, whether the goal was reached, replay updates, and run time. We will save full step-level logs and replay events for a subset of seeds, which is enough for the replay analyses, and save the SR at the end of each phase.


One limitation is that simulated data only reflects the environments we design. Maze size, slip rate, and where the change happens could all affect which agent looks best, so we will report these settings and vary them where time allows.


# Proposed Solution

We propose an agent that combines three key neuroscience-inspired principles: a predictive map (the successor representation) [<sup>[2]</sup>](#dayannote), Dyna-style offline replay from an episodic buffer [<sup>[9]</sup>](#dynanote), and a prioritized memory rule that determines which relevant memories should be accessed at each time to enable the most rewarding future decision [<sup>[8]</sup>](#mattarnote). The successor representation (SR) stores the layout of the maze, so a change in reward is understood rapidly. Prioritized replay then spends a limited number of backups on the experiences (relevant memories) that help the most, which is what the SR needs to recover from a change in layout.

### Off-policy experience resampling (SR-Dyna)
Since replay needs action values, we use the state-action version of the successor representation, $H$, from Russek et al. [<sup>[5]</sup>](#russeknote). $H(sa, s'a')$ is the expected discounted number of future visits to state-action pair $s'a'$ after taking $a$ in $s$:

$$H(sa,s'a')=\mathbb{E}\Big[\sum_{t=0}^{\infty}\gamma^{t}\,\mathbb{I}(sa_t=s'a')\Big|sa_0=sa\Big].$$

Action values are $H$ weighted by the reward of each state-action pair, where $\pi$ is the policy being followed:

$$Q^{\pi}(sa)=\sum_{s'a'}H(sa,s'a')R(s'a').$$

$H$ can be used as a linear basis for learning $Q(s,a)$, using the SARSA TD algorithm to learn a weight for each column of $H$. When state-action $s’a’$ is performed after state action $sa$, a prediction error is calculated and used to update $w$:

$$\delta=R(sa)+\gamma Q(s'a')-Q(sa)$$

$$w(i)\leftarrow w(i)+\alpha_{TD}\,\delta H(sa,i),\ \text{for all } i$$

$$H(sa,\cdot)\leftarrow H(sa,\cdot)+\alpha_{SR}\big[\mathbf{1}_{sa}+\gamma H(s'a',\cdot)-H(sa,\cdot)\big]$$

When the reward changes, only $w$ needs to change, which is why the SR adapts quickly. During offline replay, each stored sample $(s,a,s')$ updates $H$ off-policy, using the best action it could have chosen, rather than the one it actually chose:

$$H(sa,\cdot)\leftarrow H(sa,\cdot)+\alpha_{SR}\big[\mathbf{1}_{sa}+\gamma H(s'a'^{*},\cdot)-H(sa,\cdot)\big]$$

$$a'^{*}=\arg\max_{a'}\sum_{s''a''}H(s'a',s''a'')w(s''a'')$$

### Prioritized Replay
Let the agent be in state $S_t=s$ at time $t$. Each real step is stored as an experience $e_k=(s_k,a_k,r_k,s'_k)$ in the buffer, and we consider that accessing experience $e_k$ amounts to a Bellman backup which updates $Q(s_k,a_k)$ with the target value $r_k+\gamma\max_{a\in A}Q(s'_k,a)$. Following Mattar and Daw [<sup>[8]</sup>](#mattarnote), the value of replaying an experience is its expected value of backup (EVB). EVB is the improvement in expected return due to the policy change, where $\pi_{\text{old}}$ is the policy before the backup and $\pi_{\text{new}}$ is the policy after it:

$$EVB(s_k,a_k)=\mathop{\mathbb{E}}_{\pi_{new}}\Big[\sum_{i=0}^{\infty}\gamma^{i}R_{t+i}\Big|S_t=s\Big]-\mathop{\mathbb{E}}_{\pi_{old}}\Big[\sum_{i=0}^{\infty}\gamma^{i}R_{t+i}\Big|S_t=s\Big]$$

This expression separates into the product of two terms:

$$EVB(s_k,a_k)=Gain(s_k,a_k)\times Need(s_k)$$

Gain quantifies the expected improvement in return accrued at the target state $s_k$, where $\pi_{new}(a|s_k)$ and $\pi_{old}(a|s_k)$ are the probabilities of selecting action $a$ in $s_k$ after and before the Bellman backup:

$$Gain(s_k,a_k)=\sum_{a\in A}Q_{\pi_{new}}(s_k,a)\pi_{new}(a|s_k)-\sum_{a\in A}Q_{\pi_{new}}(s_k,a)\pi_{old}(a|s_k)$$

Need is the discounted number of times the agent is expected to visit the target state (a proxy for the current relevance of each state), starting from its current state $S_t=s$, where $\delta_{\cdot,\cdot}$ is the Kronecker delta function:

$$Need(s_k)=\sum_{i=0}^{\infty}\gamma^{i}\delta_{S_{t+i},s_k}$$

The need term can be estimated by the SR, being obtained directly from the $n$th row of the SR matrix.

### Algorithm
Mattar and Daw use a softmax policy; however, we will use the same $\epsilon$-greedy rule for every agent so the comparison is fair and consistent; thus, at each real step the agent acts $\epsilon$-greedily, stores $(s,a,r,s')$ in the buffer, and applies the online TD updates. It then performs $k$ replay backups, each time replaying the stored experience with the highest EVB and updating its SR row. Need is computed once per step, and EVB is only recomputed for experiences whose Gain may have changed. The main comparison uses one-step backups, and we add the $n$-step (sequence-extending) backup from Mattar and Daw specifically to test forward replay (H3).

### Benchmarks and ablations
Our benchmarks are tabular Q-learning (model-free), Dyna-Q with uniform replay [<sup>[9]</sup>](#dynanote), value iteration on a learned model (model-based), SR-TD [<sup>[2]</sup>](#dayannote), and SR-Dyna with uniform replay [<sup>[5]</sup>](#russeknote). SR-TD isolates the effect of replay, and SR-Dyna (uniform) isolates prioritization. Ablations vary the replay budget ($k \in \{0, 5, 20, 50\}$), the discount factor, and the exploration noise. We also remove the SR by running prioritized replay on a plain Q-table, which is the original Mattar and Daw agent. If time permits, we will also explore distributional and DQN agents.

### Implementation and testing
The mazes will be custom `gymnasium.Env` classes (reward revaluation, transition revaluation, and Tolman detour), and the code will be entirely in Python with the following libraries: NumPy, SciPy, pandas, statsmodels, and matplotlib. Hyperparameters ($\alpha_{SR}$, $\alpha_{TD}$, $\gamma$, $\epsilon$, $k$) are tuned by grid search with an equal budget per agent, using seeds that are separate from the evaluation seeds. Before running experiments, we will check that the online SR converges to its closed-form value on small mazes and reproduces known results from Russek et al. (SR-TD passes reward revaluation, yet fails detour tasks).

### Feasibility and rationale
The SR separates the maze structure from the reward, so a reward change only requires updating $w$. A layout change is the SR’s known weakness, and our proposed fix is to re-learn $H$ offline. Prioritization spends the limited backup budget at experiences where both Need and Gain are high (where it changes behavior most). In their open-field and linear-track grid-worlds, Mattar and Daw showed this ordering (the sequence in which stored experiences are replayed) reduced the number of steps needed to reach the reward on each trial faster than no replay and random replay, but only in stationary, deterministic tasks (while explicitly noting nonstationarity as a limitation). Testing the rule when the environment changes is the major new and exploratory aspect of our project.

## Agents Overview

| Agent | Bio-inspired component | Role | Relevant hypotheses |
|---|---|---|---|
| Tabular Q-learning | Temporal-difference learning, analogous to dopaminergic reward prediction errors [<sup>[10]</sup>](#schultznote) | Model-free baseline; slow to adapt because it must re-experience a change | H1 |
| Dyna-Q (uniform replay) | Offline replay of stored experiences in random order, with no prioritization [<sup>[9]</sup>](#dynanote) | Replay baseline with no SR and no prioritization | H4 |
| Model-based planner (value iteration on learned model) | Planning over a learned model of the maze, as in cognitive-map planning [<sup>[1]</sup>](#suttonnote) | Reference for fast adaptation at high computational cost | H2 |
| SR-TD | Predictive map of expected future state visits, resembling hippocampal place cells [<sup>[2]</sup>](#dayannote) [<sup>[4]</sup>](#stachenfeldnote) | SR baseline with no replay; expected to adapt to reward changes but not layout changes | H1, H2 |
| SR-Dyna (uniform replay) | Predictive map & offline replay of experienced transitions [<sup>[5]</sup>](#russeknote) | Ablation that isolates the effect of prioritization | H1, H2, H4 |
| Prioritized replay on a Q-table | Utility-driven replay (Need x Gain) with no predictive map [<sup>[8]</sup>](#mattarnote) | Ablation that removes the SR (the original Mattar and Daw agent) | H3, H4 |
| **SR + prioritized replay (need x gain)** | Predictive map & utility-driven replay [<sup>[5]</sup>](#russeknote) [<sup>[8]</sup>](#mattarnote) | **Main agent** | **H2, H3, H4** |
| Distributional variant of main agent (if we have enough time) | Distributional code for value in dopamine neurons [<sup>[11]</sup>](#dabneynote) | Population coding of dopamine, extension | None (exploratory) |
| DQN with experience replay (if we have enough time) | Neocortical slow learner trained from a fast replay buffer (Complementary Learning Systems) [<sup>[7]</sup>](#clsnote) | Deep RL comparison, extension | None (exploratory) |

# Evaluation Metrics

We will measure every agent in the same way, and we will only use the episodes that happen after the maze has been changed.

### Scoring normalized agent performance

After each post-change episode, we pause the agent and check how good its current strategy actually is. To do this, we look at the path it would take if it always picked the move it currently thinks is best, without doing any random exploring. Instead of just running that strategy one time and writing down the result, which would be pretty noisy, we calculate the exact reward the agent should expect to get from the start state, which we write as $J_e$. This calculation also includes the 10% chance that the agent slips into a different cell than the one it chose.

We then compare this number to two reference points. The first one, $J^{*}$, is the best possible reward an agent could get in the changed maze. The second one, $J^{\text{rand}}$, is the reward an agent would get if it just moved completely at random. Using these two numbers, we turn the agent's reward into a score that falls between 0 and 1:

$$\rho_e=\frac{J_e-J^{\text{rand}}}{J^{*}-J^{\text{rand}}}$$

A score of 0 means that the agent is doing no better than if it were moving randomly, and a score of 1 means that it is doing as well as it possibly could. This is the same basic idea that Mnih et al.<a name="mnih"></a>[<sup>[14]</sup>](#mnihnote) used, except that they compared their agents to human players, while we are comparing our agents to the best possible strategy. Putting everything on a 0 to 1 scale also lets us compare results across our different tasks, even when the best possible reward is not the same in each one.

### Behavior and efficiency
- Recovery time: this is how many episodes it takes, after the maze changes, for the agent's score to reach 0.9 when we average it over the last 10 episodes. We will also report this in terms of the total number of steps the agent took. If an agent never reaches 0.9 within the 200 post-change episodes, we will mark it as not recovered and report how often this ends up happening.
- Area under the learning curve (AUC): this is the agent's average score across all 200 of the post-change episodes. It shows how well the agent did overall, instead of only telling us the point where it crossed 0.9.
- Regret: this measures how much reward the agent misses out on while it is still learning and exploring. For each episode, we take the best possible reward and subtract the reward that the agent is expected to get with how it is currently behaving, including any random exploring. We then add these differences up across all of the episodes:
$$\text{Regret}(E)=\sum_{e=1}^{E}\big(J^{*}-V^{\pi_e^{\text{beh}}}(s_0)\big)$$
A lower regret means that the agent wasted less reward while it was adapting to the change. This measure is based on the definition of regret given in Lattimore and Szepesvári<a name="lattimore"></a>[<sup>[15]</sup>](#lattimorenote).
- Compute: this is how much work each agent has to do, which we measure using the total number of replay updates, the number of updates it needs before it recovers, and how long it takes to run. We plan to plot recovery time against the total number of updates so we can see which agents are able to adapt quickly without needing a large amount of computation.

### Neuroscience-oriented metrics
- Replay direction: we will label each replay event as either forward or reverse, following the approach used by Mattar and Daw [<sup>[8]</sup>](#mattarnote). Forward replay moves along a path in the same direction the agent would actually travel, while reverse replay moves backward from where the agent ended up. We will only count sequences of five or more replays in a row that pass a permutation test. We will then compare how much of the replay is forward right before the agent starts moving versus right after it gets a reward. Our hypothesis H3 predicts that there will be more forward replay before the agent starts moving.
- Replay content: this is how much of the replay happens along the path that actually matters for the task, compared with how much we would expect if replay just picked states at random.
- SR shape: for each cell in the maze, the SR tells us how often the agent expects to visit every other cell in the future. If we draw this out as a heat map on top of the maze, Stachenfeld et al. [<sup>[4]</sup>](#stachenfeldnote) showed that it should look similar to a hippocampal place field, which is basically a smooth blob of activity centered around one area. We will measure how smooth and clustered these maps are using Moran's $I$<a name="moran"></a>[<sup>[13]</sup>](#morannote), which is a standard way of checking whether cells that are close to each other tend to have similar values:
$$I=\frac{N}{W}\frac{\sum_i\sum_j w_{ij}(x_i-\bar{x})(x_j-\bar{x})}{\sum_i(x_i-\bar{x})^2}$$
In this formula, $x_i$ is the SR value at cell $i$, $\bar{x}$ is the average SR value across all cells, $w_{ij}$ is 1 if cells $i$ and $j$ are next to each other and 0 if they are not, $N$ is the total number of cells, and $W$ is the total number of neighboring pairs. A score close to 1 means the map is smooth, like a place field, and a score close to 0 means it looks more or less random. If we end up having enough time, we will also look at the SR's eigenvectors, which Stachenfeld et al. linked to grid cells, but we would only compare these by looking at them rather than measuring them.

### Statistics
To check whether the differences between agents are real and not just due to chance, we will use bootstrap confidence intervals along with nonparametric tests, which are tests that do not assume our data follows a normal distribution. We will use a Kruskal-Wallis test to compare all of the agents at once, and then use Mann-Whitney $U$ tests to compare pairs of agents, with a Holm correction<a name="holm"></a>[<sup>[12]</sup>](#holmnote) to account for the fact that we are running many tests. For comparisons that come from the same runs, such as forward replay before moving versus after getting a reward, we will use the Wilcoxon signed-rank test. We will report effect sizes along with $p$-values, and we will post our hypotheses H1 to H4 in our repository before we run the final experiments.

# Ethics & Privacy

Our project has limited privacy and ethical concerns, since we solely use data from our simulated environments, instead of real-world data which could possibly include personal information. However, if we do choose to include animal or human behavioral datasets, we will ensure that the data is publicly available, licensed as needed, and even de-identified as required.

One potential ethical concern is bias in training data and experience replay. Since our reinforcement learning agents prioritize certain experiences during learning, they may overrepresent more frequently rewarded states while other less common behaviors are less represented. We’ll evaluate this through different conditions in the state space, and introduce stochasticity in our learning environment.

We will avoid overstating biological relevance in our findings as well, since similarities between agent behavior and hippocampal replay do not necessarily mean the same mechanisms occur in a real brain. Limitations will be documented, with positive and negative results to ensure transparency and reproducibility.


# Team Expectations

Our team will adhere to the following expectations:
* Clear communication is not only established but utilized extensively through Discord or iMessage
* Bi-Weekly check-ins, increasing in frequency as needed and especially in before important submissions
* Decision with project direction will be majority ruled, with major decision backed by reason and facts (and reality based on timeline and feasibility)


# Project Timeline Proposal

Task assignment will be delegated before and/or at meetings.


| Meeting Date  | Meeting Time| Completed Before Meeting  | Discuss at Meeting |
|---|---|---|---|
| 11/6  |  1 PM |  Literature review of models and brainstorming evaluation metrics (All)  | Compare notes and understanding of models, identify data sets, start architecting the state space and virtual environment |
| 11/9  |  12 PM |  Identify and wrangle datasets (All)  | Discuss datasets and how each model needs structured data. Divide up models for implementation |
| 11/13  | 12 PM  | Implementation of models divided up between each group member. Identify any questions or issues to discuss at the meeting (All)  | Issues with implementation, data structuring/wrangling, etc. Continue to evaluate and analyze |
| 11/16  | 12 PM  | Continued implementation of models, any and all questions addressed offline (All) | Analysis and evaluation work session, ensure project requirements are completed until this point. If any large questions, email TA and/or prof then |
| 11/20  | 12 PM  | Last minute implementation and model tests, analysis completed (All) | Touch base before Thanksgiving, ensure bulk of the project is done |
| 12/1  | 12 PM  | Complete analysis; Draft results/conclusion/discussion (All) | Discuss/edit full project |
| 12/5  | Before 11:59 PM  | Double check requirements, ensure project is complete (All) | Turn in Final Project |

# Footnotes

<a name="suttonnote"></a>1.[^](#sutton): Sutton, R. S. (1991). Dyna, an integrated architecture for learning, planning, and reacting. *SIGART Bulletin*, 2(4). https://doi.org/10.1145/122344.122377<br>
<a name="dayannote"></a>2.[^](#dayan): Dayan, P. (1993). Improving generalization for temporal difference learning: The successor representation. *Neural Computation*, 5(4). https://doi.org/10.1162/neco.1993.5.4.613<br>
<a name="momennejadnote"></a>3.[^](#momennejad): Momennejad, I., Russek, E. M., Cheong, J. H., Botvinick, M. M., Daw, N. D., & Gershman, S. J. (2017). The successor representation in human reinforcement learning. *Nature Human Behaviour*. https://doi.org/10.1038/s41562-017-0180-8<br>
<a name="stachenfeldnote"></a>4.[^](#stachenfeld): Stachenfeld, K. L., Botvinick, M. M., & Gershman, S. J. (2017). The hippocampus as a predictive map. *Nature Neuroscience*. https://doi.org/10.1038/nn.4650<br>
<a name="russeknote"></a>5.[^](#russek): Russek, E. M., Momennejad, I., Botvinick, M. M., Gershman, S. J., & Daw, N. D. (2017). Predictive representations can link model-based reinforcement learning to model-free mechanisms. *PLOS Computational Biology*. https://doi.org/10.1371/journal.pcbi.1005768<br>
<a name="fosternote"></a>6.[^](#foster): Foster, D. J., & Wilson, M. A. (2006). Reverse replay of behavioural sequences in hippocampal place cells during the awake state. *Nature*. https://doi.org/10.1038/nature04587<br>
<a name="clsnote"></a>7.[^](#cls): McClelland, J. L., McNaughton, B. L., & O'Reilly, R. C. (1995). Why there are complementary learning systems in the hippocampus and neocortex. *Psychological Review*. Also: Kumaran, D., Hassabis, D., & McClelland, J. L. (2016). What learning systems do intelligent agents need? *Trends in Cognitive Sciences*.<br>
<a name="mattarnote"></a>8.[^](#mattar): Mattar, M. G., & Daw, N. D. (2018). Prioritized memory access explains planning and hippocampal replay. *Nature Neuroscience*. https://doi.org/10.1038/s41593-018-0232-z<br>
<a name="dynanote"></a>9.[^](#dyna): Sutton (1991), see reference 1.<br>
<a name="schultznote"></a>10.[^](#schultz): Schultz, W., Dayan, P., & Montague, P. R. (1997). A neural substrate of prediction and reward. *Science*, 275(5306). https://doi.org/10.1126/science.275.5306.1593<br>
<a name="dabneynote"></a>11.[^](#dabney): Dabney, W., Kurth-Nelson, Z., Uchida, N., et al. (2020). A distributional code for value in dopamine-based reinforcement learning. *Nature*. https://doi.org/10.1038/s41586-019-1924-6<br>
<a name="holmnote"></a>12.[^](#holm): Holm, S. (1979). A simple sequentially rejective multiple test procedure. *Scandinavian Journal of Statistics*, 6(2). https://www.jstor.org/stable/4615733<br>
<a name="morannote"></a>13.[^](#moran): Cliff, A. D., & Ord, J. K. (1973). *Spatial Autocorrelation*. London: Pion. (General weighted form of Moran's I, with its moments under randomization and normality.) Originally introduced in: Moran, P. A. P. (1948). The interpretation of statistical maps. *Journal of the Royal Statistical Society, Series B*, 10(2), 243–251.<br>
<a name="mnihnote"></a>14.[^](#mnih): Mnih, V., Kavukcuoglu, K., Silver, D., et al. (2015). Human-level control through deep reinforcement learning. *Nature*, 518, 529–533. https://doi.org/10.1038/nature14236<br>
<a name="lattimorenote"></a>15.[^](#lattimore): Lattimore, T., & Szepesvári, C. (2020). *Bandit Algorithms*. Cambridge University Press. https://doi.org/10.1017/9781108571401<br>

# AI Usage Acknowledgement

AI-assistance was employed for translation of some variables and formulas into markdown LaTeX and the formatting of some citations mentioned in the Background section. All AI usage was checked against the necessary literature for correctness.